

---


# Clustering Movies from desctiptions using vector embeddings

---



In [3]:
from scipy.cluster.hierarchy import linkage , dendrogram
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering,KMeans
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

# Tis will not work in surprise environment

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
df = pd.read_csv('titles.csv')
df.shape

(5850, 15)

In [ ]:
df

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5845,tm1014599,Fine Wine,MOVIE,A beautiful love story that can happen between...,2021,NaN,100,"['romance', 'drama']",['NG'],NaN,tt13857480,6.8,45.0,1.466,NaN
5846,tm898842,C/O Kaadhal,MOVIE,A heart warming film that explores the concept...,2021,NaN,134,['drama'],[],NaN,tt11803618,7.7,348.0,NaN,NaN
5847,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021,NaN,90,['comedy'],['CO'],NaN,tt14585902,3.8,68.0,26.005,6.300
5848,tm1035612,Dad Stop Embarrassing Me - The Afterparty,MOVIE,"Jamie Foxx, David Alan Grier and more from the...",2021,PG-13,37,[],['US'],NaN,NaN,NaN,NaN,1.296,10.000


### Filtering Indian movies

In [ ]:
df_indian_movies = df[(df['production_countries'].str.contains('IN')) & (df['type']=='MOVIE')]
df_indian_movies

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
19,tm94651,Dostana,MOVIE,Two close friends decide to enter law enforcem...,1980,NaN,161,"['drama', 'comedy', 'crime', 'romance', 'action']",['IN'],NaN,tt0080653,2.1,25.0,3.980,4.9
21,tm100027,Alibaba Aur 40 Chor,MOVIE,The movie is based on the story of Ali Baba an...,1979,NaN,146,"['drama', 'romance', 'action', 'fantasy', 'fam...","['SU', 'IN']",NaN,tt0079749,6.2,565.0,2.582,6.8
25,tm356209,Singapore,MOVIE,"When his employee disappears in Singapore, Shy...",1960,NaN,158,"['drama', 'thriller', 'crime']",['IN'],NaN,tt0268639,6.4,84.0,0.792,6.3
29,tm336403,Khoon Khoon,MOVIE,As a crazed killer blazes a trail of blood thr...,1973,NaN,131,"['crime', 'thriller']",['IN'],NaN,tt0230412,5.1,41.0,1.948,3.8
30,tm107372,Manoranjan,MOVIE,Havaldar Ratan is a rookie at the local police...,1974,NaN,162,"['romance', 'drama', 'comedy']",['IN'],NaN,tt0071811,6.5,237.0,1.532,5.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5766,tm1004011,Time to Dance,MOVIE,When a ballroom dancer’s shot at a crucial tou...,2021,NaN,2,"['romance', 'drama']",['IN'],NaN,tt8622232,2.2,963.0,1.043,4.0
5788,tm987730,Tribhanga,MOVIE,"When her estranged mother falls into a coma, a...",2021,NaN,95,"['drama', 'family']",['IN'],NaN,tt11102314,6.2,2922.0,4.294,7.0
5795,tm988613,Madam Chief Minister,MOVIE,A political-drama where a young woman from a s...,2021,NaN,123,['drama'],['IN'],NaN,tt13773882,4.8,1769.0,2.301,6.7
5821,tm1110241,Kaaval,MOVIE,Thampan and Antony are a long time best friend...,2021,NaN,148,"['thriller', 'action', 'drama']",['IN'],NaN,tt11182984,5.1,1497.0,1.978,5.3


In [ ]:
model  = SentenceTransformer('BAAI/bge-small-en-v1.5')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
model.encode("I am feeling excellent").shape

(384,)

### Every embeded text will have array of 384 numbers

In [ ]:
all_desc = list(df_indian_movies['description'])
out_embs = []
for r in tqdm(all_desc):
  embd = model.encode(r)
  out_embs.append(embd)
out_embd = np.array(out_embs)

100%|██████████| 577/577 [00:42<00:00, 13.47it/s]


In [ ]:
out_embd.shape

(577, 384)

In [ ]:
df_indian_movies['movieID']= df_indian_movies['id']+'-'+df_indian_movies['title']
df_embd = pd.DataFrame(out_embs,index = list(df_indian_movies['movieID']))
df_embd

/tmp/ipykernel_1300/2353834856.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_indian_movies['movieID']= df_indian_movies['id']+'-'+df_indian_movies['title']


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383
tm94651-Dostana,-0.075357,0.013868,-0.061348,-0.009090,-0.004694,0.005166,0.089489,-0.046013,0.048869,-0.018892,0.030726,0.015970,0.029219,-0.073880,0.084885,0.026612,0.001922,-0.033890,-0.116655,0.098042,-0.042150,-0.026964,-0.010000,-0.014489,-0.070391,0.094976,0.019533,-0.018994,-0.036232,-0.023085,-0.026810,-0.006642,0.040608,0.055823,0.029376,0.016031,0.056253,-0.013799,0.036768,-0.008343,...,-0.038681,0.078037,-0.001330,0.041037,-0.016712,-0.015012,-0.037460,0.084524,-0.085204,0.043361,-0.012456,0.011395,0.083137,0.056902,0.029572,0.031346,-0.018417,-0.026792,-0.077373,0.008848,-0.011331,-0.018206,0.054651,-0.073227,0.080940,-0.018060,0.040422,-0.028759,-0.053822,0.008427,0.022732,-0.026808,0.006294,0.038266,-0.005323,-0.062236,0.033453,0.020772,0.049413,0.022834
tm100027-Alibaba Aur 40 Chor,-0.044140,0.060243,-0.041446,-0.058401,-0.030420,0.064920,0.065799,-0.036992,0.012539,-0.077255,0.028708,-0.027167,-0.002213,0.013110,-0.012943,-0.015417,0.057614,-0.028853,-0.059884,-0.000650,0.041652,0.015718,0.036442,0.017018,0.005415,-0.075608,0.036446,-0.006139,-0.068389,-0.086220,-0.043451,-0.015965,0.059691,0.018752,-0.044674,0.015665,-0.015894,0.038155,-0.035542,-0.016185,...,-0.022021,-0.023619,-0.058001,-0.095772,-0.016795,-0.018558,0.032417,0.088227,0.015756,0.024700,0.013201,0.043964,0.011154,0.069306,0.038455,-0.015185,0.027517,-0.026553,-0.052475,0.038064,-0.028542,-0.026392,0.061559,-0.017234,0.011082,-0.042399,-0.005455,0.068928,0.016134,0.011158,0.031550,-0.054891,0.015133,-0.024666,-0.019042,-0.131373,0.014510,0.012415,-0.009698,0.019127
tm356209-Singapore,0.033862,0.030536,-0.017178,-0.036037,0.088664,-0.030931,0.148762,0.001592,0.019263,-0.022572,0.014480,0.007618,0.091768,0.063016,0.021250,-0.018986,-0.017095,-0.057464,-0.006173,-0.054839,0.005865,-0.058513,-0.000989,-0.107580,-0.011806,-0.016475,0.016788,-0.006381,-0.037816,-0.048172,0.015705,-0.021297,-0.027319,0.022093,0.054172,0.021640,-0.014623,0.010644,0.084733,-0.008542,...,0.009546,0.042197,0.035729,0.022506,-0.022293,0.035299,-0.097886,0.019867,0.014036,-0.014967,-0.027936,-0.040332,0.048457,0.039917,0.001004,-0.021294,-0.030155,0.001961,-0.068011,0.054241,-0.036286,-0.107028,0.047314,-0.042221,-0.007258,-0.044055,-0.012537,0.061911,-0.018841,0.020499,0.023382,-0.013798,-0.014579,0.031583,-0.072774,0.019934,0.000184,0.010525,-0.041923,-0.008498
tm336403-Khoon Khoon,-0.149428,0.020323,0.018530,0.007980,0.055575,0.028446,0.104850,-0.024781,-0.041700,-0.006586,0.021891,-0.027322,-0.061462,-0.032638,-0.030421,0.031249,0.076111,0.043277,-0.025589,0.076298,-0.045018,0.047053,0.028907,-0.056367,-0.016634,-0.017755,0.057896,0.082013,-0.056197,-0.023224,0.022140,-0.037630,0.010658,0.014279,-0.002191,-0.014382,-0.083988,0.072239,-0.008057,0.035683,...,0.065171,0.033967,-0.033672,0.009545,-0.049898,0.024594,-0.017101,-0.012674,-0.007888,0.042827,0.053644,-0.083727,0.058871,0.076526,0.015094,-0.031759,0.055171,0.018352,-0.028680,0.032696,0.022804,-0.143191,0.069365,-0.012023,0.051625,-0.008668,0.011665,0.008222,-0.016832,0.067855,0.063270,-0.055972,-0.006201,0.087120,0.032169,-0.029733,0.059928,0.019334,0.041777,-0.091707
tm107372-Manoranjan,-0.044192,0.066234,-0.047135,-0.032663,0.065942,-0.011514,0.099225,-0.020301,-0.015988,0.024434,0.023420,-0.014965,-0.008578,0.027971,0.016500,-0.011704,0.005124,-0.022464,-0.032337,0.059997,0.031789,-0.088869,-0.031372,-0.061799,0.016771,-0.030964,0.052787,0.005833,-0.069261,-0.109550,0.048840,-0.003508,0.081183,0.054818,0.038569,-0.025845,-0.023586,0.019394,0.061321,0.019675,...,0.027319,0.047474,-0.042755,-0.020418,-0.045623,-0.020064,-0.022999,0.032794,-0.029497,-0.008983,-0.021570,0.005304,0.068372,0.010143,0.005002,-0.007493,0.053570,0.004170,0.011495,0

In [ ]:
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_samples

In [ ]:
hdbbscan = HDBSCAN(min_cluster_size=2)
hdbbscan.fit(out_embs)
df_embd['hdbscan_cluster'] = hdbbscan.labels_
df_embd['hdbscan_cluster'].value_counts()

,count
hdbscan_cluster,
-1,413
57,20
40,9
58,6
48,5
25,4
36,4
27,4
51,3


In [ ]:
df_embd[df_embd['hdbscan_cluster']==40]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,hdbscan_cluster
tm358653-Ujala,-0.049333,0.046700,-0.023438,0.011097,0.056499,-0.009629,0.074433,-0.050439,-0.009558,-0.041754,0.025846,-0.002995,0.021458,0.021007,0.011030,0.068221,-0.016217,0.045389,-0.064676,0.011731,-0.046714,-0.051835,0.029118,-0.031425,0.020721,0.033621,0.028327,0.001513,-0.062212,0.017308,0.003888,-0.006660,0.060626,-0.028748,-0.022162,-0.004400,-0.046360,-0.008204,-0.042937,-0.006454,...,0.061335,-0.020625,0.005027,-0.055691,-0.020811,0.063503,0.035798,-0.036503,0.005576,0.012827,-0.030081,0.048311,0.028278,0.009083,-0.071451,0.013870,-0.034530,0.042348,0.074266,-0.048846,0.010165,0.053720,0.010176,0.005203,-0.008597,0.026482,0.014456,-0.005800,0.012316,0.044418,-0.051632,-0.018721,0.103138,-0.037706,-0.060866,0.070764,0.029892,-0.000398,-0.086344,40
tm36475-Kabhi Khushi Kabhie Gham,-0.009770,0.098484,-0.033455,-0.068486,-0.000378,-0.069111,-0.009649,0.004997,-0.016360,-0.051542,0.075535,0.052723,0.053374,-0.039003,0.099233,0.053344,-0.049222,0.125126,-0.096751,-0.026073,-0.067052,-0.052147,-0.026054,0.007239,0.037503,0.088311,0.021652,0.019645,-0.000772,-0.025241,0.082996,0.058729,0.017295,0.031414,0.099125,-0.023749,-0.085899,-0.000880,-0.079359,0.048770,...,0.066121,-0.050378,-0.005103,-0.069086,0.008341,0.037117,0.037691,0.023657,0.047176,-0.000797,-0.024001,0.022350,0.030548,0.029546,-0.102546,0.057334,0.008688,-0.039138,0.065409,-0.052101,-0.006748,0.038165,0.011386,0.015147,-0.013689,0.047982,-0.019060,0.008470,0.053708,0.011661,-0.088768,0.042016,0.056744,-0.031425,-0.004988,0.057972,0.013204,-0.049761,-0.032099,40
tm201365-Brothers,-0.059697,0.039701,-0.078245,-0.021248,0.027391,0.021967,0.020873,-0.060083,0.033656,-0.003242,0.050778,-0.060456,0.038716,0.037086,0.070467,0.045628,-0.047110,0.044659,-0.058275,0.024506,-0.088619,-0.017380,0.024257,-0.009631,0.045070,0.062320,-0.033834,0.032174,-0.029036,-0.098901,0.094232,0.013045,0.029315,0.012838,-0.012517,-0.044418,0.008840,-0.058148,-0.018585,-0.024481,...,-0.024620,-0.071658,-0.001675,-0.063003,-0.080543,0.032330,0.099452,-0.030509,0.002638,-0.001092,-0.076348,0.019304,0.005360,0.018880,-0.047774,-0.064187,-0.057150,-0.056381,0.076239,0.063746,-0.012654,0.020836,-0.018062,0.012619,-0.050845,0.037562,0.008671,-0.007437,-0.051277,0.025968,0.019473,-0.015799,0.036002,-0.069868,0.030096,0.058978,-0.033188,-0.003713,-0.078360,40
tm233344-Kapoor & Sons,0.004743,0.063174,-0.035938,-0.020779,0.033685,0.004174,0.013526,-0.011958,-0.035909,-0.068807,0.018683,0.031899,0.024887,-0.019292,0.072071,0.032719,-0.066747,0.110457,-0.095627,0.031893,-0.090391,-0.047377,0.035607,0.010593,-0.003788,0.076826,-0.001114,0.044537,-0.002306,-0.053145,0.021369,0.015700,-0.006286,0.049019,0.056516,-0.007582,-0.013336,0.002350,-0.029038,-0.064688,...,0.028259,-0.003784,-0.029912,-0.017826,-0.044772,-0.026683,0.034317,-0.018649,0.071566,-0.032715,-0.018473,0.018632,0.025202,-0.005168,-0.091942,-0.026254,-0.072262,-0.019315,0.056768,-0.008272,-0.025988,0.082113,-0.032509,-0.018974,-0.003999,0.076683,0.007814,-0.027951,0.074296,-0.053940,-0.009119,0.015855,0.084987,-0.034363,0.007396,0.086275,0.034912,-0.066075,-0.026168,40
tm431928-Rajma Chawal,0.013039,0.035680,-0.024108,-0.060963,0.000311,-0.009750,0.013484,-0.001609,-0.030677,-0.071405,0.052670,0.050440,0.021157,0.030720,0.120004,0.071610,0.001047,0.004259,-0.077670,0.020183,-0.018440,-0.009301,0.018916,-0.027384,-0.011500,0.001577,0.018135,0.018114,-0.027227,-0.038665,0.058363,-0.028717,0.019151,0.025347,0.027587,-0.043516,-0.019382,-0.013569,-0.073897,0.032850,...,0.028528,-0.027619,0.021260,-0.013723,-0.028650,0.009630,0.018533,0.017939,0.062668,-0.052502,-0.013915,-0.022990,0.002024,-0.037535,-0.033257,0.058364,-0.050688,-0.012682,0.065084,0.006919,

In [ ]:
df_indian_movies[(df_embd['hdbscan_cluster']==48).values]

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,movieID
434,tm61056,Dil Chahta Hai,MOVIE,Three inseparable childhood friends are just o...,2001,G,184,"['romance', 'comedy', 'drama']",['IN'],NaN,tt0292490,8.1,71586.0,5.704,7.0,tm61056-Dil Chahta Hai
566,tm154375,Main Aurr Mrs Khanna,MOVIE,"Three people, each having different aspiration...",2009,PG,115,"['drama', 'romance', 'comedy']",['IN'],NaN,tt1174041,3.4,3173.0,2.810,3.9,tm154375-Main Aurr Mrs Khanna
721,tm182125,Zindagi Na Milegi Dobara,MOVIE,Three friends who were inseparable in childhoo...,2011,PG,166,"['romance', 'drama', 'comedy']",['IN'],NaN,tt1562872,8.1,76565.0,11.997,7.6,tm182125-Zindagi Na Milegi Dobara
1455,tm233843,Pyaar Ka Punchnama 2,MOVIE,"After falling in love, three roommates experie...",2015,PG-13,159,"['drama', 'romance', 'comedy']",['IN'],NaN,tt4430136,7.3,15216.0,3.924,7.0,tm233843-Pyaar Ka Punchnama 2
4248,tm946277,Love,MOVIE,The story of a family and the various situatio...,2020,NaN,91,"['thriller', 'drama']",['IN'],NaN,tt12573294,7.0,1544.0,2.055,5.9,tm946277-Love
